<a href="https://colab.research.google.com/github/1jsrayner-cmd/cascade-seismic-3d/blob/main/Copy_of_st_helens_quakes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Where under the volcano do earthquakes happen, and how deep?

In [ ]:
!pip install py3dep

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

In [ ]:
LAT0, LON0 = 46.200, -122.189      # approximate summit
HALF_LAT, HALF_LON = 0.10, 0.15    # half-size of the box in degrees
south, north = LAT0 - HALF_LAT, LAT0 + HALF_LAT
west,  east  = LON0 - HALF_LON, LON0 + HALF_LON

In [ ]:
url = (
    "https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv"
    "&starttime=2004-01-01&endtime=2026-10-01&eventtype=earthquake"
    f"&minlatitude={south}&maxlatitude={north}"
    f"&minlongitude={west}&maxlongitude={east}"
    "&minmagnitude=1.0&orderby=time"
)
quakes = pd.read_csv(url, parse_dates=["time"])

In [ ]:
quakes.head()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2026-09-29 13:24:21.620000+00:00,46.204000,-122.186500,2.88,1.64,md,35,51.0,0.003864,0.08,...,2026-09-29T18:23:07.760Z,"38 km NNE of Amboy, Washington",earthquake,0.17,0.22,0.272714,14.0,reviewed,uw,uw
1,2026-09-27 11:43:32.840000+00:00,46.205667,-122.192000,2.79,1.00,mh,9,84.0,0.005941,0.06,...,2026-09-28T18:16:26.620Z,"38 km NNE of Amboy, Washington",earthquake,0.28,0.66,NaN,0.0,reviewed,uw,uw
2,2026-09-26 00:17:45.970000+00:00,46.199333,-122.183500,0.28,2.01,md,11,80.0,0.001637,0.14,...,2026-09-28T20:47:18.640Z,"38 km NNE of Amboy, Washington",earthquake,0.40,0.23,0.191416,4.0,reviewed,uw,uw
3,2026-09-20 19:09:30.320000+00:00,46.203167,-122.193333,-1.58,1.29,md,10,60.0,0.003799,0.06,...,2026-09-21T21:40:18.680Z,"38 km NNE of Amboy, Washington",earthquake,0.29,2.06,0.422413,6.0,reviewed,uw,uw
4,2026-09-20 03:26:33.440000+00:00,46.199333,-122.196167,-1.78,1.51,md,17,58.0,0.003715,0.18,...,2026-09-21T19:46:29.520Z,"37 km NNE of Amboy, Washington",earthquake,0.24,14.98,0.136977,6.0,reviewed,uw,uw


In [ ]:
import plotly.express as px
px.scatter(quakes, x="longitude", y="latitude", color="depth").show()

In [ ]:
px.histogram(quakes, x="depth").show()

In [ ]:
per_year = quakes.groupby(quakes["time"].dt.year).size()
px.bar(x=per_year.index, y=per_year.values).show()

In [ ]:
quakes = quakes.dropna(subset=["depth"])

In [ ]:
import py3dep
dem = py3dep.get_map("DEM", (west, south, east, north), resolution=60,
                     geo_crs="EPSG:4326", crs="EPSG:4326")
dem = dem.sortby("y")

In [ ]:
print(dem)

<xarray.DataArray 'elevation' (y: 340, x: 511)> Size: 695kB
array([[ 498.79663,  505.13553,  507.56747, ...,  501.27213,  501.27213,
         497.5633 ],
       [ 505.65646,  507.17847,  507.97357, ...,  556.5013 ,  529.2758 ,
         525.6463 ],
       [ 506.82977,  505.85785,  505.85785, ...,  578.0826 ,  567.79913,
         541.3818 ],
       ...,
       [1139.5275 , 1139.5275 , 1136.5779 , ...,  965.9712 ,  993.735  ,
        1022.4242 ],
       [1155.8407 , 1149.6432 , 1139.3844 , ...,  969.76074,  994.32684,
        1018.7729 ],
       [1164.2544 , 1159.6174 , 1146.416  , ..., 1003.1362 , 1003.1362 ,
        1031.29   ]], dtype=float32)
Coordinates:
  * y            (y) float64 3kB 46.1 46.1 46.1 46.1 ... 46.3 46.3 46.3 46.3
  * x            (x) float64 4kB -122.3 -122.3 -122.3 ... -122.0 -122.0 -122.0
    spatial_ref  int64 8B 0
Attributes:
    DataType:         Generic
    AREA_OR_POINT:    Area
    SourceBandIndex:  0
    scale_factor:     1.0
    add_offset:       0.0
    _F

In [ ]:
float(dem.min()), float(dem.max())

(394.9364318847656, 2528.73681640625)

In [ ]:
px.imshow(dem.values, origin="lower").show()

In [ ]:
!pip install rasterio

In [ ]:
kx = 111.32 * np.cos(np.radians(LAT0))   # km per degree of longitude
ky = 110.95                              # km per degree of latitude

In [ ]:
X = (dem.x.values - LON0) * kx
Y = (dem.y.values - LAT0) * ky
Z = dem.values / 1000

In [ ]:
qx = (quakes["longitude"] - LON0) * kx
qy = (quakes["latitude"]  - LAT0) * ky
qz = -quakes["depth"]

In [ ]:
marker=dict(size=2, color=quakes["time"].dt.year,
            colorscale="Plasma", colorbar=dict(title="Year"))

In [ ]:
hover = (
    "Date: " + quakes["time"].dt.strftime("%Y-%m-%d %H:%M") + " UTC<br>"
    + "Magnitude: " + quakes["mag"].astype(str) + "<br>"
    + "Depth: " + quakes["depth"].round(1).astype(str) + " km<br>"
    + quakes["place"].astype(str)
)

In [ ]:
dots = go.Scatter3d(
    x=qx, y=qy, z=qz, mode="markers",
    marker=dict(size=3, color="red"),
    text=hover,
    hovertemplate="%{text}<extra></extra>",
    name="Earthquakes",
)

In [ ]:
terrain = go.Surface(x=X, y=Y, z=Z, colorscale="Earth",
                     opacity=0.6, showscale=False, name="Terrain")

dots = go.Scatter3d(x=qx, y=qy, z=qz, mode="markers",
                    marker=dict(size=2, color="red"), name="Earthquakes")

fig = go.Figure(data=[terrain, dots])
fig.update_layout(
    title="Earthquakes beneath Mount St. Helens",
    scene=dict(xaxis_title="East (km)", yaxis_title="North (km)",
               zaxis_title="Elevation / depth (km)", aspectmode="data"),
)
fig.show()